# Unified Segmentation RDM - Debug Playground

This notebook helps debug the Unified Segmentation RDM training pipeline.

**Data Paths:**
- Images: `/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train`
- SAM Embeddings: `/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/{class_id}/masks_npz/*.npz`
- IJEPA Embeddings: `/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/output_dir/ijepa_embeddings/{class_id}/*.npz`

## 1. Setup Environment

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

# Add repo to path
repo_root = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Verify Data Paths and Structure

In [ ]:
# Define all data paths
IMAGENET_DIR = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train"
SAM_CACHE_DIR = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified"
IJEPA_EMB_DIR = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/output_dir/ijepa_embeddings"

# Check if paths exist
print("Checking data paths...")
print(f"ImageNet train: {Path(IMAGENET_DIR).exists()} - {IMAGENET_DIR}")
print(f"SAM cache: {Path(SAM_CACHE_DIR).exists()} - {SAM_CACHE_DIR}")
print(f"IJEPA embeddings: {Path(IJEPA_EMB_DIR).exists()} - {IJEPA_EMB_DIR}")

# Count classes
if Path(IMAGENET_DIR).exists():
    imagenet_classes = sorted([d.name for d in Path(IMAGENET_DIR).iterdir() if d.is_dir()])
    print(f"\nImageNet classes: {len(imagenet_classes)}")
    print(f"First 5 classes: {imagenet_classes[:5]}")
    
if Path(SAM_CACHE_DIR).exists():
    sam_classes = sorted([d.name for d in Path(SAM_CACHE_DIR).iterdir() if d.is_dir()])
    print(f"\nSAM cache classes: {len(sam_classes)}")
    print(f"First 5 classes: {sam_classes[:5]}")
    
if Path(IJEPA_EMB_DIR).exists():
    ijepa_classes = sorted([d.name for d in Path(IJEPA_EMB_DIR).iterdir() if d.is_dir()])
    print(f"\nIJEPA embedding classes: {len(ijepa_classes)}")
    print(f"First 5 classes: {ijepa_classes[:5]}")

## 3. Inspect Data Format

Check the structure of SAM and IJEPA embeddings.

In [ ]:
# Pick a sample class (e.g., class 0)
sample_class = "0"

# === Check ImageNet images ===
imagenet_class_dir = Path(IMAGENET_DIR) / sample_class
if imagenet_class_dir.exists():
    image_files = sorted(list(imagenet_class_dir.glob("*.JPEG")))
    print(f"Class {sample_class} - ImageNet images: {len(image_files)}")
    if image_files:
        sample_img = image_files[0]
        print(f"  Sample: {sample_img.name}")
        img = Image.open(sample_img)
        print(f"  Size: {img.size}, Mode: {img.mode}")

# === Check SAM embeddings ===
sam_class_dir = Path(SAM_CACHE_DIR) / sample_class / "masks_npz"
if sam_class_dir.exists():
    sam_files = sorted(list(sam_class_dir.glob("*.npz")))
    print(f"\nClass {sample_class} - SAM embeddings: {len(sam_files)}")
    if sam_files:
        sample_npz = sam_files[0]
        print(f"  Sample: {sample_npz.name}")
        data = np.load(sample_npz)
        print(f"  Keys: {list(data.keys())}")
        for key in data.keys():
            arr = data[key]
            if hasattr(arr, 'shape'):
                print(f"    {key}: shape={arr.shape}, dtype={arr.dtype}")
            else:
                print(f"    {key}: {type(arr)}")

# === Check IJEPA embeddings ===
ijepa_class_dir = Path(IJEPA_EMB_DIR) / sample_class
if ijepa_class_dir.exists():
    ijepa_files = sorted(list(ijepa_class_dir.glob("*.npz")))
    print(f"\nClass {sample_class} - IJEPA embeddings: {len(ijepa_files)}")
    if ijepa_files:
        sample_npz = ijepa_files[0]
        print(f"  Sample: {sample_npz.name}")
        data = np.load(sample_npz)
        print(f"  Keys: {list(data.keys())}")
        for key in data.keys():
            arr = data[key]
            if hasattr(arr, 'shape'):
                print(f"    {key}: shape={arr.shape}, dtype={arr.dtype}")
            else:
                print(f"    {key}: {type(arr)}")

## 4. Test Dataset Loader

Test the `SegmentationMaskDataset` with your actual data paths.

In [ ]:
from rdm.data.seg_dataset import SegmentationMaskDataset, collate_seg_batch
from torch.utils.data import DataLoader

# Create dataset with ImageNet structure
# Note: Dataset expects flat structure, but ImageNet has class folders
# We'll test with a single class first

test_image_dir = Path(IMAGENET_DIR) / "0"  # Test with class 0
test_sam_dir = Path(SAM_CACHE_DIR) / "0" / "masks_npz"

print(f"Testing dataset with:")
print(f"  Images: {test_image_dir}")
print(f"  SAM embeddings: {test_sam_dir}")

dataset = SegmentationMaskDataset(
    image_dir=str(test_image_dir),
    mask_npz_dir=str(test_sam_dir),
    max_segments=250,
    image_size=256,
    file_ext="*.JPEG",  # ImageNet uses .JPEG
    normalize=True,
)

print(f"\nDataset size: {len(dataset)}")

if len(dataset) > 0:
    # Test single sample
    sample = dataset[0]
    print(f"\nSample 0:")
    for key, val in sample.items():
        if isinstance(val, torch.Tensor):
            print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
        else:
            print(f"  {key}: {val}")
    
    # Test dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=4,
        shuffle=True,
        num_workers=0,  # 0 for debugging
        collate_fn=collate_seg_batch
    )
    
    batch = next(iter(dataloader))
    print(f"\nBatch:")
    for key, val in batch.items():
        if isinstance(val, torch.Tensor):
            print(f"  {key}: shape={val.shape}, dtype={val.dtype}")
        elif isinstance(val, list):
            print(f"  {key}: list of {len(val)} items")
        else:
            print(f"  {key}: {type(val)}")
    
    print("\n✓ Dataset test passed!")
else:
    print("❌ Dataset is empty! Check file paths.")

## 5. Visualize Sample Data

Visualize an image with its SAM segmentation masks.

In [ ]:
if len(dataset) > 0:
    # Get a sample
    sample = dataset[0]
    
    # Denormalize image from [-1, 1] to [0, 1]
    image = (sample['image'] * 0.5 + 0.5).permute(1, 2, 0).numpy()
    image = np.clip(image, 0, 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot image
    axes[0].imshow(image)
    axes[0].set_title(f"Image: {sample['filename']}")
    axes[0].axis('off')
    
    # Plot segmentation embedding heatmap
    seg_embs = sample['seg_embs'][:sample['num_segments']]  # Only valid segments
    axes[1].imshow(seg_embs.T.numpy(), aspect='auto', cmap='viridis')
    axes[1].set_title(f"SAM Embeddings ({sample['num_segments']} segments)")
    axes[1].set_xlabel('Segment Index')
    axes[1].set_ylabel('Embedding Dimension (256)')
    
    # Plot scores distribution
    scores = sample['scores'].numpy()
    axes[2].hist(scores, bins=20, edgecolor='black')
    axes[2].set_title(f"Score Distribution")
    axes[2].set_xlabel('Confidence Score')
    axes[2].set_ylabel('Count')
    axes[2].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nStatistics:")
    print(f"  Num segments: {sample['num_segments']}")
    print(f"  Score range: [{scores.min():.3f}, {scores.max():.3f}]")
    print(f"  Score mean: {scores.mean():.3f}")
    print(f"  Embedding norm: {seg_embs.norm(dim=1).mean():.3f}")

## 6. Load IJEPA Embeddings

Test loading IJEPA embeddings for the same images.

In [ ]:
# Load IJEPA embedding for the same sample
if len(dataset) > 0:
    sample_name = dataset[0]['filename']
    
    # Find corresponding IJEPA embedding
    ijepa_path = Path(IJEPA_EMB_DIR) / "0" / f"{sample_name}.npz"
    
    if ijepa_path.exists():
        ijepa_data = np.load(ijepa_path)
        print(f"IJEPA embedding for {sample_name}:")
        for key in ijepa_data.keys():
            arr = ijepa_data[key]
            if hasattr(arr, 'shape'):
                print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")
                if key == 'embedding' and len(arr.shape) == 1:
                    print(f"    Norm: {np.linalg.norm(arr):.3f}")
                    print(f"    Mean: {arr.mean():.3f}, Std: {arr.std():.3f}")
    else:
        print(f"❌ IJEPA embedding not found: {ijepa_path}")
        print(f"   Expected path structure: {IJEPA_EMB_DIR}/{{class_id}}/{{image_stem}}.npz")

## 7. Check Data Alignment

Verify that SAM and IJEPA embeddings exist for the same images.

In [ ]:
# Check alignment for class 0
test_class = "0"

imagenet_files = set([f.stem for f in (Path(IMAGENET_DIR) / test_class).glob("*.JPEG")])
sam_files = set([f.stem for f in (Path(SAM_CACHE_DIR) / test_class / "masks_npz").glob("*.npz")])
ijepa_files = set([f.stem for f in (Path(IJEPA_EMB_DIR) / test_class).glob("*.npz")])

print(f"Alignment check for class {test_class}:")
print(f"  ImageNet images: {len(imagenet_files)}")
print(f"  SAM embeddings: {len(sam_files)}")
print(f"  IJEPA embeddings: {len(ijepa_files)}")

# Check intersection
sam_overlap = imagenet_files & sam_files
ijepa_overlap = imagenet_files & ijepa_files
full_overlap = imagenet_files & sam_files & ijepa_files

print(f"\nOverlap:")
print(f"  Images with SAM: {len(sam_overlap)} ({len(sam_overlap)/len(imagenet_files)*100:.1f}%)")
print(f"  Images with IJEPA: {len(ijepa_overlap)} ({len(ijepa_overlap)/len(imagenet_files)*100:.1f}%)")
print(f"  Images with both: {len(full_overlap)} ({len(full_overlap)/len(imagenet_files)*100:.1f}%)")

# Show missing files
missing_sam = imagenet_files - sam_files
missing_ijepa = imagenet_files - ijepa_files

if missing_sam:
    print(f"\n❌ {len(missing_sam)} images missing SAM embeddings")
    print(f"   First 5: {list(missing_sam)[:5]}")
    
if missing_ijepa:
    print(f"\n❌ {len(missing_ijepa)} images missing IJEPA embeddings")
    print(f"   First 5: {list(missing_ijepa)[:5]}")

if len(full_overlap) == len(imagenet_files):
    print(f"\n✓ Perfect alignment! All images have both SAM and IJEPA embeddings.")
elif len(full_overlap) > 0:
    print(f"\n⚠️  Partial alignment. Training will use {len(full_overlap)} images with complete data.")
else:
    print(f"\n❌ No alignment! No images have both embeddings.")

## 8. Test Model Initialization

Load the config and test model instantiation.

In [ ]:
from omegaconf import OmegaConf
from rdm.util import instantiate_from_config

# Load config
config_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/configs/unified_seg_rdm.yaml"
config = OmegaConf.load(config_path)

print("Config loaded:")
print(OmegaConf.to_yaml(config))

# Update paths in config
config.model.params.seg_npz_dir = str(Path(SAM_CACHE_DIR) / "0" / "masks_npz")
config.data.params.image_dir = str(Path(IMAGENET_DIR) / "0")
config.data.params.mask_npz_dir = str(Path(SAM_CACHE_DIR) / "0" / "masks_npz")

print("\nUpdated paths:")
print(f"  seg_npz_dir: {config.model.params.seg_npz_dir}")
print(f"  image_dir: {config.data.params.image_dir}")
print(f"  mask_npz_dir: {config.data.params.mask_npz_dir}")

In [ ]:
# Try to instantiate model (might fail if dependencies are missing)
try:
    print("Instantiating model...")
    model = instantiate_from_config(config.model)
    print(f"✓ Model created successfully")
    print(f"  Model type: {type(model).__name__}")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
    # Move to GPU if available
    if torch.cuda.is_available():
        model = model.cuda()
        print(f"  ✓ Model moved to GPU")
        
except Exception as e:
    print(f"❌ Model instantiation failed: {e}")
    import traceback
    traceback.print_exc()

## 9. Test Forward Pass

Run a dummy forward pass to check if everything works.

In [ ]:
if 'model' in locals() and len(dataset) > 0:
    try:
        print("Testing forward pass...")
        model.eval()
        
        # Get a batch
        dataloader = DataLoader(
            dataset,
            batch_size=2,
            shuffle=False,
            num_workers=0,
            collate_fn=collate_seg_batch
        )
        batch = next(iter(dataloader))
        
        # Move to GPU
        if torch.cuda.is_available():
            batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
        
        print(f"  Batch shapes:")
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                print(f"    {k}: {v.shape}")
        
        # Forward pass
        with torch.no_grad():
            loss, loss_dict = model(x=None, c=None, batch=batch)
        
        print(f"\n✓ Forward pass successful!")
        print(f"  Loss: {loss.item():.4f}")
        print(f"  Loss dict: {loss_dict}")
        
    except Exception as e:
        print(f"❌ Forward pass failed: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Skipping forward pass test (model not loaded or dataset empty)")

## 10. Launch Training (Debug Mode)

Start training with minimal steps for debugging.

In [ ]:
# Set up environment for single-GPU training
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"

# Update config for debugging
config.training.max_steps = 50
config.training.log_every_n_steps = 5
config.training.save_checkpoint_every_n_steps = 25
config.data.batch_size = 2
config.data.num_workers = 0
config.training.use_fp16 = False  # Disable for clearer debugging

# Set output directory
output_dir = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/debug_output"
config.training.checkpoint_dir = output_dir
config.logging.tensorboard_dir = f"{output_dir}/tensorboard"
config.logging.use_wandb = False  # Disable wandb for debugging

print("Debug configuration:")
print(f"  max_steps: {config.training.max_steps}")
print(f"  batch_size: {config.data.batch_size}")
print(f"  output_dir: {output_dir}")
print(f"\nReady to train! Uncomment the cell below to start.")

In [ ]:
# # Uncomment to start training
# from train_unified_seg_rdm import UnifiedSegRDMTrainer

# # Save debug config
# debug_config_path = f"{output_dir}/debug_config.yaml"
# os.makedirs(output_dir, exist_ok=True)
# OmegaConf.save(config, debug_config_path)

# # Create trainer
# trainer = UnifiedSegRDMTrainer(
#     config_path=debug_config_path,
#     resume_from=None
# )

# # Start training
# trainer.train()

## 11. CLI Command Reference

Commands to run training from command line.

### Basic Training Command

```bash
python train_unified_seg_rdm.py \
    --config rdm/configs/unified_seg_rdm.yaml \
    --image_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train/0 \
    --sam_cache_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified/0/masks_npz \
    --ijepa_emb_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/output_dir/ijepa_embeddings/0 \
    --output_dir checkpoints/debug_run \
    --batch_size 4 \
    --max_steps 100 \
    --num_workers 0
```

### Full ImageNet Training (All Classes)

```bash
python train_unified_seg_rdm.py \
    --config rdm/configs/unified_seg_rdm.yaml \
    --image_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train \
    --sam_cache_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified \
    --ijepa_emb_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/output_dir/ijepa_embeddings \
    --output_dir checkpoints/unified_seg_rdm \
    --batch_size 32 \
    --max_steps 500000 \
    --num_workers 8 \
    --use_fp16
```

### Multi-GPU Training

```bash
torchrun --nproc_per_node=4 train_unified_seg_rdm.py \
    --config rdm/configs/unified_seg_rdm.yaml \
    --image_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train \
    --sam_cache_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/sam_cache_unified \
    --ijepa_emb_dir /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/output_dir/ijepa_embeddings \
    --output_dir checkpoints/unified_seg_rdm \
    --batch_size 16 \
    --max_steps 500000 \
    --num_workers 8 \
    --use_fp16
```

### SLURM Submission

```bash
sbatch rdm/submit_rdm_train.sh
```